# bag0 v5: loss-variant gradients at grid 144 on the two squeeze parts
f64 adjoints (one per part and loss variant). GPU memory: ~16 GB per adjoint on the host at grid 48; use an A100 or L4 runtime, a T4 (15 GB) will likely run out of memory. Mount Drive with `bag0_dynamics_v5_gpu_package.zip` in MyDrive root. Outputs: `MyDrive/bag0_colab/grad_v5g144.jsonl` (one JSON row per run) and per-run logs.

In [ ]:
BRANCH = 'bounded-shear-adhesion-v3'
GRID = 144
PARTS = ['bag0_p06_E1.6_squeeze', 'bag0_p07_E2.1_squeeze']
VARIANTS = [('baseline', 'ident', []), ('shape_rel', 'shape', ['--relative']), ('shape_rel_late', 'shape', ['--relative', '--late']), ('shape_rel_plus_orig', 'shape', ['--relative', '--orig-weight', '1.0'])]

In [ ]:
!nvidia-smi -L
!pip -q install taichi==1.7.4 open3d==0.20.0 scipy
import taichi, open3d; print(taichi.__version__, open3d.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bag0_colab
!ls -la /content/drive/MyDrive/bag0_dynamics_v5_gpu_package.zip

In [ ]:
%cd /content
!rm -rf TaichiDough && git clone -q --branch $BRANCH https://github.com/AntonioSkara132/TaichiDough.git
%cd /content/TaichiDough
!git log --oneline -1
import zipfile
z = zipfile.ZipFile('/content/drive/MyDrive/bag0_dynamics_v5_gpu_package.zip')
z.extractall('/content/TaichiDough', [n for n in z.namelist() if n.startswith('experiments/')])
print('extracted')

In [ ]:
# same local patches as the GPU host: cache identity version pins; SDF tolerance pinned to the grid-48 value
import pathlib
p = pathlib.Path('experiments/differentiable_mpm/sdf_cache.py'); s = p.read_text()
old = '    return hashlib.sha256(_json(description).encode()).hexdigest(), description'
new = '\n'.join([
    '    def _pin(v):',
    '        if isinstance(v, dict): return {k: ("2.2.6" if k == "numpy_version" else "0.17.0" if k == "open3d_version" else _pin(x)) for k, x in v.items()}',
    '        if isinstance(v, list): return [_pin(x) for x in v]',
    '        return v',
    '    description = _pin(description)',
    old])
assert old in s; p.write_text(s.replace(old, new, 1))
p = pathlib.Path('experiments/differentiable_mpm/data.py'); s = p.read_text()
old = '            settings = qualification_settings(config.tool_sdf_qualification, grid_spacing)'
new = '            grid_spacing = 1.0 / 48  # tolerance pinned to grid 48 (shipped 256^3 fields)\n' + old
assert old in s; p.write_text(s.replace(old, new, 1)); print('patched')

In [ ]:
# grid-144 box configs (box indices x3)
import json
D = 'experiments/differentiable_mpm/data/bag0_dynamics_v5/'
d = json.load(open(D + 'dataset_bag0_all_von_mises_1000pa_coulomb_v5.json')); f = GRID // 48; eps = []
for e in d['episodes']:
    c = json.load(open(D + e['config'])); c['simulation']['grid'] = GRID
    c['simulation']['grid_box_lower'] = [v * f for v in c['simulation']['grid_box_lower']]; c['simulation']['grid_box_cells'] = [v * f for v in c['simulation']['grid_box_cells']]
    new = e['config'].replace('_coulomb_v5.json', '_coulomb_v5_g%d.json' % GRID); json.dump(c, open(D + new, 'w'), indent=2); eps.append(dict(e, config=new))
dd = dict(d); dd['name'] += '-grid%dbox' % GRID; dd['episodes'] = eps; json.dump(dd, open(D + 'dataset_v5_g%d.json' % GRID, 'w'), indent=2); print('ok')

In [ ]:
!mkdir -p /content/scripts

In [ ]:
%%writefile /content/scripts/shape_loss.py
"""Shape-descriptor loss wrapper for TaichiDough rollouts (no change to the experiment tree).

Descriptors per frame: soft width along x, soft width along z (soft max - soft min, temperature tau), soft height
(soft max of y). Recorded descriptors come from the recorded clouds in the scene frame. The loss is the mean over
scored frames of sum_d ((d_sim - d_rec) / scale)^2 (optionally on changes relative to the first scored frame, which
cancels a constant reconstruction offset), weighted per frame, plus `orig_weight` times the original observation
loss. Returns LossRecord(value, dL/dpositions, components, diagnostics) like ObservationLoss."""
import numpy as np

def _band_mean(c, lo_frac, hi_frac, sign=1.0):
    """Mean of the values ranked between lo_frac and hi_frac from the top (sign=+1) or bottom (sign=-1); gradient 1/k on
    the band members (piecewise linear, robust to the extreme (1-hi_frac) fraction of outliers)."""
    n = c.shape[0]; order = np.argsort(sign * c); lo, hi = int(lo_frac * n), int(hi_frac * n)  # ascending order of sign*c: top band = ranks [lo_frac, hi_frac)
    idx = order[lo:hi]; k = len(idx); m = float(c[idx].mean()); g = np.zeros_like(c); g[idx] = 1.0 / k
    return m, g

def descriptors(x, tau=None, band=(0.90, 0.975)):
    """x: (N,3) scene-frame positions. d = [width_x, width_z, height] from band means on each side; dd/dx (3,N,3)."""
    d = np.zeros(3); g = np.zeros((3, x.shape[0], 3))
    for j, axis in enumerate((0, 2)):
        mx, gmx = _band_mean(x[:, axis], *band, +1.0); mn, gmn = _band_mean(x[:, axis], *band, -1.0)
        d[j] = mx - mn; g[j, :, axis] = gmx - gmn
    mh, gh = _band_mean(x[:, 1], *band, +1.0); d[2] = mh; g[2, :, 1] = gh
    return d, g

class ShapeDescriptorLoss:
    def __init__(self, base_loss, recorded, *, tau=0.004, scale=0.01, shape_weight=1.0, orig_weight=0.0, relative=False, frame_weights=None, initial_positions=None):
        self.base, self.rec, self.tau, self.scale = base_loss, recorded, tau, scale  # rec: {frame_index: np.array(3,)}
        self.ws, self.wo, self.relative, self.fw = shape_weight, orig_weight, relative, frame_weights or {}
        self.d0_sim = descriptors(np.asarray(initial_positions, dtype=np.float64), tau)[0] if (relative and initial_positions is not None) else None
        self.d0_rec = recorded.get(0) if relative else None

    def value_and_grad_positions(self, positions, observation, compute_grad=True):
        x = np.asarray(positions, dtype=np.float64); fi = int(observation.frame_index)
        d, g = descriptors(x, self.tau); target = self.rec[fi]
        if self.relative: d = d - self.d0_sim; target = target - self.d0_rec
        r = (d - target) / self.scale; w = self.fw.get(fi, 1.0)
        shape_value = w * float((r * r).sum()); grad = (w * 2.0 / self.scale) * np.einsum("k,kni->ni", r, g) if compute_grad else None
        components = {"shape": shape_value, "width_x_mm": float(d[0] * 1000), "width_z_mm": float(d[1] * 1000), "height_mm": float(d[2] * 1000), "rec_width_x_mm": float(target[0] * 1000), "rec_width_z_mm": float(target[1] * 1000), "rec_height_mm": float(target[2] * 1000)}
        value = self.ws * shape_value; diagnostics = {"loss_version": "shape-descriptors-v1", "frame_index": fi, "step": int(observation.step), "frame_weight": w}
        gradient = self.ws * grad if compute_grad else None
        if self.wo > 0:
            rec = self.base.value_and_grad_positions(positions, observation, compute_grad=compute_grad)
            value += self.wo * rec.value; components.update({"orig_" + k: v for k, v in rec.components.items()}); diagnostics.update({"orig_" + k: v for k, v in rec.diagnostics.items()})
            if compute_grad: gradient = gradient + self.wo * np.asarray(rec.gradient, dtype=np.float64)
        from experiments.differentiable_mpm.loss import LossRecord
        return LossRecord(float(value), None if gradient is None else gradient.astype(np.asarray(positions).dtype), components, diagnostics)

if __name__ == "__main__":  # finite-difference check of the descriptor gradient
    rng = np.random.default_rng(0); x = rng.normal(size=(2000, 3)) * 0.03; tau = None
    d, g = descriptors(x, tau); eps = 1e-7; err = 0
    for _ in range(40):
        i, a = rng.integers(2000), rng.integers(3); xp = x.copy(); xp[i, a] += eps; dp = descriptors(xp, tau)[0]
        num = (dp - d) / eps; err = max(err, np.abs(num - g[:, i, a]).max())
    print("descriptors (mm):", np.round(d * 1000, 2), "| max |FD - analytic| gradient error:", err)


In [ ]:
%%writefile /content/scripts/grad_ident.py
"""Gradient of each episode's calibration objective w.r.t. all fitted parameters at a given parameter point.
usage: grad_ident.py CODE_ROOT DATASET_JSON OUT_JSONL [--point JSON] [--precision f32|f64] [EPISODE_ID ...]
Rows: episode, membership, point, value, gradient (physical units), scaled = p * dL/dp, finite flags, steps, seconds."""
import json, os, sys, time, math, argparse
ap = argparse.ArgumentParser(); ap.add_argument("root"); ap.add_argument("dataset"); ap.add_argument("out"); ap.add_argument("--point", default=None); ap.add_argument("--precision", default=None); ap.add_argument("--tag", default="initial"); ap.add_argument("--stride", type=int, default=1); ap.add_argument("--depth-reference", default=None); ap.add_argument("episodes", nargs="*")
a = ap.parse_args(); os.chdir(a.root); sys.path.insert(0, a.root)
from experiments.differentiable_mpm.calibrate import make_rollout
from experiments.differentiable_mpm.data import prepare_experiment
from experiments.differentiable_mpm.dataset_config import load_dataset
from experiments.differentiable_mpm.reference_adapter import reference_policy
from experiments.differentiable_mpm.runtime import init_runtime
done = set()
if os.path.exists(a.out):
    for line in open(a.out):
        r = json.loads(line); done.add((r["episode"], r["tag"]))
quiet = lambda *x, **k: None
with reference_policy("frozen"):
    ds = load_dataset(a.dataset, backend="cuda", precision=a.precision, p2g_mode="atomic")
    shared = dict(ds.shared_initial)
    if a.point: shared.update(json.loads(a.point))
    eps = [e for e in ds.episodes if not a.episodes or e.id in a.episodes]
    first = eps[0].config; prec = a.precision or first.simulation.get("precision", "f32")
    init_runtime("cuda", prec, cpu_threads=1, seed=first.seed)
    print(f"{len(eps)} episodes, precision {prec}, point {json.dumps(shared)}", flush=True)
    for ep in eps:
        if (ep.id, a.tag) in done: continue
        t0 = time.perf_counter(); row = {"episode": ep.id, "membership": ep.membership, "tag": a.tag, "point": shared, "precision": prec, "stride": a.stride, "depth_reference": a.depth_reference}
        try:
            window = ep.scored_window if a.stride == 1 else type(ep.scored_window)(ep.scored_window.start_frame, ep.scored_window.end_frame, a.stride)
            cfg = ep.config
            if a.depth_reference:
                import dataclasses; cfg = dataclasses.replace(cfg, loss={**cfg.loss, "depth_reference": a.depth_reference}) if isinstance(cfg.loss, dict) else cfg
            prepared = prepare_experiment(cfg, split=ep.membership, build_sdf=True, scored_window=window)
            _, rollout = make_rollout(prepared, progress=quiet)
            row["steps"] = prepared.total_steps; row["prepare_s"] = round(time.perf_counter() - t0, 1); t1 = time.perf_counter()
            params = ep.parameters_for(shared)
            ev = rollout.value_and_gradient(params, compute_grad=True)
            g = {k: float(v) for k, v in (ev.gradient or {}).items()}
            row.update(value=float(ev.value), gradient=g, scaled={k: (shared.get(k, params.get(k, 0.0)) * v if isinstance(shared.get(k, params.get(k)), (int, float)) else None) for k, v in g.items()},
                       finite=all(math.isfinite(v) for v in g.values()) and math.isfinite(float(ev.value)), grad_s=round(time.perf_counter() - t1, 1),
                       diagnostics={k: ev.diagnostics.get(k) for k in ev.diagnostics if any(t in k for t in ("branch", "finite", "replay", "mismatch", "nan"))})
        except Exception as exc:
            row.update(error=f"{type(exc).__name__}: {exc}"[:400], seconds=round(time.perf_counter() - t0, 1))
        with open(a.out, "a") as f: f.write(json.dumps(row) + "\n")
        print(f"{ep.id} [{ep.membership}] value={row.get('value')} finite={row.get('finite')} grad={ {k: f'{v:.3g}' for k, v in row.get('gradient', {}).items()} } {row.get('error', '')} ({row.get('grad_s', row.get('seconds'))} s)", flush=True)


In [ ]:
%%writefile /content/scripts/grad_ident_shape.py
"""Gradient of a part's objective under the shape-descriptor loss (wrapper around the rollout's loss).
usage: grad_ident_shape.py ROOT DATASET OUT EPISODE --shape-weight W --orig-weight W [--relative] [--late] [--tag T]"""
import json, os, sys, time, math, argparse, numpy as np, torch
ap = argparse.ArgumentParser(); ap.add_argument("root"); ap.add_argument("dataset"); ap.add_argument("out"); ap.add_argument("episode")
ap.add_argument("--shape-weight", type=float, default=1.0); ap.add_argument("--orig-weight", type=float, default=0.0); ap.add_argument("--relative", action="store_true"); ap.add_argument("--late", action="store_true"); ap.add_argument("--tag", default="shape"); ap.add_argument("--precision", default="f64"); ap.add_argument("--forward-only", action="store_true")
a = ap.parse_args(); os.chdir(a.root); sys.path.insert(0, a.root); sys.path.insert(0, "/content/scripts")
from experiments.differentiable_mpm.calibrate import make_rollout
from experiments.differentiable_mpm.data import prepare_experiment
from experiments.differentiable_mpm.dataset_config import load_dataset
from experiments.differentiable_mpm.reference_adapter import reference_policy
from experiments.differentiable_mpm.runtime import init_runtime
from shape_loss import ShapeDescriptorLoss, descriptors
D = os.path.dirname(a.dataset) + "/"; T = np.asarray(json.load(open(D + "scene_calibration_v2.json"))["scene_from_source"])
with reference_policy("frozen"):
    ds = load_dataset(a.dataset, backend="cuda", precision=a.precision, p2g_mode="atomic"); ep = next(e for e in ds.episodes if e.id == a.episode)
    init_runtime("cuda", a.precision, cpu_threads=1, seed=ep.config.seed)
    t0 = time.perf_counter(); prepared = prepare_experiment(ep.config, split=ep.membership, build_sdf=True, scored_window=ep.scored_window); _, rollout = make_rollout(prepared, progress=lambda *x, **k: None)
    clouds = torch.load(D + f"recordings/{a.episode}/pointclouds_interpolated.pt", map_location="cpu", weights_only=True)[0]
    rec = {}
    for i in range(len(clouds)):
        p = np.asarray(clouds[i])[:, :3].astype(np.float64); p = p[np.isfinite(p).all(1)] @ T[:3, :3].T + T[:3, 3]; rec[i] = descriptors(p, 0.004)[0]
    frames = sorted(o.frame_index for o in rollout.observations); n = len(frames)
    fw = {f: (2.0 * (k + 1) / n) for k, f in enumerate(frames)} if a.late else None  # late weighting: ramps 0->2, mean 1
    init_x = np.asarray(prepared.initial_state.x if hasattr(prepared, "initial_state") else rollout.initial_state.x, dtype=np.float64)
    rollout.loss = ShapeDescriptorLoss(rollout.loss, rec, tau=0.004, scale=0.01, shape_weight=a.shape_weight, orig_weight=a.orig_weight, relative=a.relative, frame_weights=fw, initial_positions=init_x)
    shared = dict(ds.shared_initial); params = ep.parameters_for(shared); t1 = time.perf_counter()
    try:
        ev = rollout.value_and_gradient(params, compute_grad=not a.forward_only); g = {k: float(v) for k, v in (ev.gradient or {}).items()}
        row = {"episode": ep.id, "membership": ep.membership, "tag": a.tag, "shape_weight": a.shape_weight, "orig_weight": a.orig_weight, "relative": a.relative, "late": a.late, "value": float(ev.value), "gradient": g,
               "scaled": {k: (shared.get(k, params.get(k, 0.0)) * v) for k, v in g.items()}, "finite": all(math.isfinite(v) for v in g.values()), "grad_s": round(time.perf_counter() - t1, 1), "steps": prepared.total_steps,
               "frames": [{k: v for k, v in f["components"].items()} | {"frame": f["frame_index"]} for f in ev.frames[:: max(1, len(ev.frames) // 8)]]}
    except Exception as exc:
        row = {"episode": ep.id, "tag": a.tag, "error": f"{type(exc).__name__}: {exc}"[:400]}
    with open(a.out, "a") as f: f.write(json.dumps(row) + "\n")
    print(f"{ep.id} [{a.tag}] value={row.get('value')} finite={row.get('finite')} scaled={ {k: round(v, 4) for k, v in row.get('scaled', {}).items() if k in ('youngs_modulus','poisson_ratio','viscosity','tool_friction_coefficient','floor_friction_coefficient','von_mises_yield_stress_pa')} } {row.get('error', '')}", flush=True)


In [ ]:
import os, subprocess
os.environ['TAICHIDOUGH_SDF_CACHE'] = '/content/TaichiDough/experiments/differentiable_mpm/data/bag0_dynamics_v5_sdf_cache'
ROOT = '/content/TaichiDough'; DS = 'experiments/differentiable_mpm/data/bag0_dynamics_v5/dataset_v5_g%d.json' % GRID; OUT = '/content/drive/MyDrive/bag0_colab/grad_v5g144.jsonl'
for part in PARTS:
    for tag, kind, extra in VARIANTS:
        script = '/content/scripts/grad_ident.py' if kind == 'ident' else '/content/scripts/grad_ident_shape.py'
        args = ['python', '-u', script] + (['--precision', 'f64', '--tag', tag, ROOT, DS, OUT, part] if kind == 'ident' else ['--tag', tag] + extra + [ROOT, DS, OUT, part])
        log = '/content/drive/MyDrive/bag0_colab/grad144_%s_%s.log' % (tag, part)
        with open(log, 'w') as f: r = subprocess.run(args, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        print(part, tag, 'exit', r.returncode, [l for l in open(log) if 'value=' in l or 'Error' in l][-1:][:1])

In [ ]:
import json
P = ['youngs_modulus', 'poisson_ratio', 'viscosity', 'tool_friction_coefficient', 'floor_friction_coefficient', 'von_mises_yield_stress_pa']
rows = [json.loads(l) for l in open('/content/drive/MyDrive/bag0_colab/grad_v5g144.jsonl')]
print('%-44s %7s ' % ('part / loss', 'loss') + ' '.join('%8s' % k[:8] for k in P))
for r in rows:
    if 'error' in r: print(r['episode'][5:], r['tag'], 'ERROR', r['error'][:100]); continue
    sc = [r['scaled'].get(k) if r['scaled'].get(k) is not None else r['gradient'][k] * 1000.0 for k in P]
    print('%-44s %7.3f ' % (r['episode'][5:] + ' / ' + r['tag'], r['value']) + ' '.join('%+8.3f' % v for v in sc))